# 🚗 VehiclEye — Entrenamiento en Google Colab

**Duración:** ~2 horas con GPU T4 gratis  
**Ejecuta cada celda con Shift+Enter y espera ✅ antes de continuar.**

## CELDA 1 — Verificar GPU y configurar entorno

In [ ]:
# Silenciar DeprecationWarnings de jupyter_client (no afectan el código)
import warnings
warnings.filterwarnings('ignore')

import os, sys, torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('⚠️  Sin GPU — ve a: Entorno de ejecución → Cambiar tipo → GPU T4')

print(f'✅ Python {sys.version[:6]}  |  PyTorch {torch.__version__}')

## CELDA 2 — Instalar dependencias

In [ ]:
import warnings; warnings.filterwarnings('ignore')

# Dependencias ML
!pip install -q timm albumentations
# Descarga de imágenes
!pip install -q duckduckgo-search requests pillow icrawler
# Exportación ONNX
!pip install -q onnx onnxruntime

import timm, PIL, onnx
print(f'✅ timm={timm.__version__}  PIL={PIL.__version__}  onnx={onnx.__version__}')
print('✅ Todas las dependencias instaladas.')

## CELDA 3 — Clonar repositorio

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys

REPO_URL    = 'https://github.com/nick2331/Electiva_3.git'
REPO_BRANCH = 'claude/analyze-project-tech-oKyKr'
REPO_DIR    = '/content/Electiva_3'

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
    print('✅ Repositorio clonado.')
else:
    !git -C {REPO_DIR} pull origin {REPO_BRANCH} -q
    print('✅ Repositorio actualizado.')

os.chdir(REPO_DIR)
# Agregar raíz al sys.path para que los imports de ml.* funcionen
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'📁 CWD: {os.getcwd()}')
print(f'✅ sys.path incluye {REPO_DIR}')

## CELDA 4 — Descargar imágenes de vehículos

Usa **DuckDuckGo** (primario) + **icrawler/Bing** (respaldo) con:
- Validación pixel a pixel (rechaza siluetas, anime, logos)
- Deduplicación por hash MD5
- Reintentos automáticos si se bloquea

⏱️ ~25-35 minutos

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import time, hashlib, random, requests
from pathlib import Path
from io import BytesIO
from PIL import Image

IMAGES_PER_CLASS = 60
MIN_VALID        = 25
OUTPUT_DIR       = Path('ml/data/vehicleye_dataset')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VEHICLE_CLASSES = [
    ('Toyota','Corolla'),('Toyota','Hilux'),
    ('Chevrolet','Spark'),('Chevrolet','Aveo'),
    ('Renault','Logan'),('Renault','Sandero'),('Renault','Stepway'),
    ('Mazda','3'),('Mazda','CX-5'),
    ('Hyundai','Tucson'),('Hyundai','Accent'),
    ('Kia','Picanto'),('Kia','Rio'),
    ('Nissan','Frontier'),('Nissan','Versa'),
    ('Ford','Fiesta'),('Ford','Escape'),
    ('Volkswagen','Gol'),('Volkswagen','Jetta'),
    ('Suzuki','Swift'),
]

def to_dir_name(brand, model):
    return f'{brand}_{model}'.lower().replace('-','').replace(' ','_').replace('./','')

def is_valid(data, min_px=100):
    try:
        img = Image.open(BytesIO(data)).convert('RGB')
        w, h = img.size
        if w < min_px or h < min_px: return False
        px = list(img.getdata())
        n  = len(px)
        black = sum(1 for r,g,b in px if r<40  and g<40  and b<40)
        white = sum(1 for r,g,b in px if r>220 and g>220 and b>220)
        gray  = sum(1 for r,g,b in px if abs(r-g)<12 and abs(g-b)<12)
        if black/n > 0.55: return False
        if white/n > 0.80: return False
        if gray/n  > 0.90: return False
        return True
    except: return False

def save_img(data, path):
    img = Image.open(BytesIO(data)).convert('RGB')
    img.save(path, 'JPEG', quality=90)

# ── Estrategia 1: DuckDuckGo ─────────────────────────────────────
def download_ddg(brand, model, class_dir, need):
    try:
        from duckduckgo_search import DDGS
    except: return 0
    queries = [
        f'{brand} {model} car exterior',
        f'{brand} {model} automobile side view photo',
        f'{brand} {model} vehicle parked road',
        f'{brand} {model} car 2020 2021 2022',
    ]
    done = 0; seen = set()
    for q in queries:
        if done >= need: break
        try:
            with DDGS() as ddgs:
                hits = list(ddgs.images(q, max_results=25, type_image='photo', size='Medium'))
        except Exception as e:
            time.sleep(4); continue
        for h in hits:
            if done >= need: break
            url = h.get('image','')
            if not url: continue
            try:
                r = requests.get(url, timeout=8); 
                if r.status_code != 200: continue
                md5 = hashlib.md5(r.content).hexdigest()
                if md5 in seen: continue
                seen.add(md5)
                if not is_valid(r.content): continue
                save_img(r.content, class_dir/f'img_{len(list(class_dir.glob("*.jpg"))):04d}.jpg')
                done += 1
            except: continue
        time.sleep(2)
    return done

# ── Estrategia 2: icrawler (Bing) — respaldo ─────────────────────
def download_icrawler(brand, model, class_dir, need):
    try:
        from icrawler.builtin import BingImageCrawler
        before = len(list(class_dir.glob('*.jpg')))
        crawler = BingImageCrawler(
            feeder_threads=1, parser_threads=2, downloader_threads=4,
            storage={'root_dir': str(class_dir)}
        )
        crawler.crawl(
            keyword=f'{brand} {model} car',
            filters={'type':'photo','size':'medium'},
            max_num=need, min_size=(100,100)
        )
        after = len(list(class_dir.glob('*.jpg')))
        return after - before
    except Exception as e:
        return 0

# ── Descarga una clase completa ───────────────────────────────────
def download_class(brand, model):
    dname = to_dir_name(brand, model)
    cdir  = OUTPUT_DIR / dname
    cdir.mkdir(parents=True, exist_ok=True)

    existing = len(list(cdir.glob('*.jpg')))
    if existing >= IMAGES_PER_CLASS:
        print(f'  ✅ {brand} {model}: {existing} imgs (ya listo)')
        return existing

    need = IMAGES_PER_CLASS - existing
    n1 = download_ddg(brand, model, cdir, need)
    still_need = IMAGES_PER_CLASS - len(list(cdir.glob('*.jpg')))
    n2 = 0
    if still_need > 5:
        n2 = download_icrawler(brand, model, cdir, still_need)

    total = len(list(cdir.glob('*.jpg')))
    icon = '✅' if total >= MIN_VALID else '⚠️ '
    print(f'  {icon} {brand:12} {model:10}: {total} imgs  (DDG:{n1} Bing:{n2})')
    return total

# ── Ejecutar ──────────────────────────────────────────────────────
print('📥 Descargando imágenes de vehículos...\n')
results = []
for brand, model in VEHICLE_CLASSES:
    print(f'⬇️  {brand} {model}...')
    n = download_class(brand, model)
    results.append((brand, model, n))

total   = sum(n for *_, n in results)
good    = sum(1 for *_, n in results if n >= MIN_VALID)
print(f'\n{"="*50}')
print(f'Total imágenes : {total}')
print(f'Clases OK (≥{MIN_VALID}): {good}/20')
if good >= 15:
    print('✅ Dataset suficiente — puedes entrenar.')
else:
    print('⚠️  Pocas clases completas. Ejecuta esta celda de nuevo para descargar más.')

## CELDA 5 — Entrenar EfficientNet-B0

⏱️ ~90 min con GPU T4. Puedes dejarlo corriendo.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys

# Asegura que el proyecto esté en el path
REPO = '/content/Electiva_3'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Silencia warnings en el subprocess también
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['PYTHONPATH']     = REPO

!python -W ignore ml/train.py \
    --data_dir ml/data/vehicleye_dataset \
    --epochs_phase1 5 \
    --epochs_phase2 10 \
    --output ml/checkpoints/efficientnet_b0_vehicleye.pth

from pathlib import Path
pth = Path('ml/checkpoints/efficientnet_b0_vehicleye.pth')
if pth.exists():
    print(f'\n✅ Modelo guardado: {pth.stat().st_size/1024/1024:.1f} MB')
    print('   Puedes continuar con CELDA 7 (exportar a ONNX).')
else:
    print('\n❌ No se generó el modelo. Revisa el error arriba.')

## CELDA 6 — Evaluar accuracy (opcional)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os; os.environ['PYTHONWARNINGS']='ignore'; os.environ['PYTHONPATH']='/content/Electiva_3'

!python -W ignore ml/evaluate.py

from pathlib import Path
f = Path('reports/model_metrics.txt')
if f.exists():
    print('\n📊 MÉTRICAS:\n' + '='*60)
    print(f.read_text())
else:
    print('⚠️  Sin reporte (normal si el dataset es pequeño).')

## CELDA 7 — Exportar a ONNX (formato Render)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os; os.environ['PYTHONWARNINGS']='ignore'; os.environ['PYTHONPATH']='/content/Electiva_3'

!python -W ignore ml/export_onnx.py \
    --checkpoint ml/checkpoints/efficientnet_b0_vehicleye.pth \
    --output     ml/checkpoints/vehicleye.onnx

from pathlib import Path
onnx = Path('ml/checkpoints/vehicleye.onnx')
if onnx.exists():
    print(f'\n✅ ONNX listo: {onnx.stat().st_size/1024/1024:.1f} MB')
    print('   Listo para descargar y subir a GitHub Releases.')
else:
    print('\n❌ No se generó vehicleye.onnx — ejecuta CELDA 5 primero.')

## CELDA 8 — Descargar modelo a tu computadora

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from google.colab import files
from pathlib import Path

onnx = Path('ml/checkpoints/vehicleye.onnx')
pth  = Path('ml/checkpoints/efficientnet_b0_vehicleye.pth')

if onnx.exists():
    print(f'📥 Descargando vehicleye.onnx ({onnx.stat().st_size/1024/1024:.1f} MB)...')
    files.download(str(onnx))
    print('✅ Revisa tu carpeta Descargas.')
elif pth.exists():
    print('⚠️  vehicleye.onnx no existe. Descargando el .pth como respaldo...')
    files.download(str(pth))
    print('✅ Descargado efficientnet_b0_vehicleye.pth')
    print('   Ejecuta CELDA 7 para exportar a ONNX antes de subir a GitHub.')
else:
    print('❌ Ningún modelo encontrado. Ejecuta las celdas 5 y 7 primero.')

## CELDA 9 — Pasos siguientes

In [ ]:
print("""
🎉 ¡ENTRENAMIENTO COMPLETADO!

Tienes: vehicleye.onnx en tu carpeta Descargas.

══════════════════════════════════════════════════
PASO A — Subir a GitHub Releases
══════════════════════════════════════════════════
 1. github.com/nick2331/Electiva_3/releases
 2. "Create a new release"
 3. Tag: v1.0  |  Título: Modelo VehiclEye v1.0
 4. Arrastra vehicleye.onnx
 5. "Publish release"

══════════════════════════════════════════════════
PASO B — Copiar URL del modelo
══════════════════════════════════════════════════
 Clic derecho en vehicleye.onnx → Copiar enlace
 Ejemplo:
   https://github.com/nick2331/Electiva_3/
   releases/download/v1.0/vehicleye.onnx

══════════════════════════════════════════════════
PASO C — Configurar en Render
══════════════════════════════════════════════════
 dashboard.render.com → vehicleye-api → Environment
 Key:   MODEL_DOWNLOAD_URL
 Value: (URL del paso B)
 → Save  (redeploya automáticamente)

══════════════════════════════════════════════════
VERIFICACIÓN en Admin Panel
══════════════════════════════════════════════════
 Modelo IA: ✅ "Modelo ONNX cargado en memoria"

 ¡Las predicciones ahora son REALES! 🚗✅
""")